# Tarea 2 - Intelgencia Artificial
### *Alejandro Ignacio Jara Vergara👻 - Alexis Agustín Lema González👀*

El dataset utilizado es [online shoppers intention](https://www.kaggle.com/datasets/henrysue/online-shoppers-intention) contiene registros de 12.330 sesiones de usuarios en un sitio web comercial, recopilando múltiples atributos de cada sesión. En general para esta tarea, se buscan diferentes modelos capaces de distinguir las sesiones de usuario que terminan en una compra de aquellas que no o cuanto tiempo se tarda un usuario en buscar productos, utilizando información como:

- Administrative y Administrative_Duration: Número de páginas de tipo administrativo (por ejemplo, páginas informativas de cuenta, contacto, FAQ) visitadas en la sesión y el tiempo total dedicado a ellas.

- Informational y Informational_Duration: Número de páginas de tipo informativo (por ejemplo, detalles de productos, guías) y tiempo en ellas.

- ProductRelated y ProductRelated_Duration: Número de páginas de producto vistas en la sesión y tiempo total en páginas de producto.

- BounceRates: Tasa de rebote de la sesión (en porcentaje). Es una métrica de Google Analytics que indica la proporción de visitantes que entraron al sitio y se fueron tras ver solo una página, sin interactuar más.

- ExitRates: Tasa de salida por página, también de Analytics, que representa el porcentaje de veces que una página determinada fue la última de la sesión.

- PageValues: Valor promedio de cada página que el usuario visitó antes de completar una transacción (calculado por Analytics). Esta característica da una idea del valor económico de las páginas vistas por el usuario

- SpecialDay: Indica la cercanía de la fecha de la sesión a algún día especial o festivo comercial (por ejemplo, Día de la Madre, San Valentín)

- VisitorType: Nuevo o recurrente

- TrafficType: Fuente o tipo de tráfico numerado (canal por el cual el usuario llegó al sitio, por ejemplo: tráfico directo, motores de búsqueda, anuncios específicos, etc.).

- Month: Mes del año en que tuvo lugar la sesión.

- Weekend: Valor booleano que indica si la sesión ocurrió en fin de semana.

---


Ahora, procedemos a instalar dependencias:


In [ ]:
!pip install pandas scikit-learn kaggle

### Configuración del entorno y carga de datos [📂]

- Cargamos la ApiKey de Kaggle y el dataset


In [ ]:
import pandas as pd
import kagglehub
from google.colab import files
files.upload()

!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d henrysue/online-shoppers-intention
!unzip online-shoppers-intention.zip

df = pd.read_csv('online_shoppers_intention.csv')
print(f"Dimensiones del dataset: {df.shape}")
df.head()

Saving kaggle.json to kaggle (1).json
mkdir: cannot create directory ‘/root/.kaggle’: File exists
Dataset URL: https://www.kaggle.com/datasets/henrysue/online-shoppers-intention
License(s): other
online-shoppers-intention.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  online-shoppers-intention.zip
replace online_shoppers_intention.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: online_shoppers_intention.csv  
Dimensiones del dataset: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


### Pregunta de Clasificación (no trivial)

¿Se puede predecir si un usuario realizará una compra (`Revenue = True`) en un sitio de e-commerce a partir de su comportamiento de navegación (tiempo en páginas de producto, número de páginas en secciones administrativas o informativas, tasas de rebote y salida), el tipo de visitante (nuevo o recurrente), la fuente de tráfico, el mes de la sesión y si fue fin de semana?  


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

def get_preprocessor(X):
    cat_cols = X.select_dtypes(include=["object", "bool"]).columns
    num_cols = X.select_dtypes(include=["int64", "float64"]).columns
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ])

X_clf = df.drop("Revenue", axis=1)
y_clf = df["Revenue"]
preprocessor_clf = get_preprocessor(X_clf)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, stratify=y_clf, random_state=42
)

classifiers = {
    "SVM": SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42),
    "Regresión Logística": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "KNN Classifier": KNeighborsClassifier(n_neighbors=5)
}

clf_results = []
print("🔍 Resultados de Clasificación:\n")
for name, model in classifiers.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor_clf),
        ("classifier", model)
    ])
    pipe.fit(X_train_clf, y_train_clf)
    y_pred = pipe.predict(X_test_clf)

    acc = accuracy_score(y_test_clf, y_pred)
    f1 = f1_score(y_test_clf, y_pred)
    clf_results.append((name, acc, f1))
    print(f"{name}: Accuracy = {acc:.4f}, F1-score = {f1:.4f}")

best_clf = max(clf_results, key=lambda x: x[2])
print(f"\n✅ Mejor clasificador: {best_clf[0]} (F1-score = {best_clf[2]:.4f})")

🔍 Resultados de Clasificación:

SVM: Accuracy = 0.8560, F1-score = 0.6077
Regresión Logística: Accuracy = 0.8500, F1-score = 0.6072
KNN Classifier: Accuracy = 0.8731, F1-score = 0.4943

✅ Mejor clasificador: SVM (F1-score = 0.6077)


### Pregunta de Regresión (no trivial)

¿Es posible estimar cuánto tiempo (en segundos) pasará un usuario en la sección de productos (`ProductRelated_Duration`) de un sitio de e-commerce, basándose en su comportamiento de navegación (número y duración de visitas en secciones administrativas e informativas, tasas de rebote y salida), el tipo de visitante (nuevo o recurrente), la fuente de tráfico, el mes de la sesión y si fue fin de semana?  

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

target_reg = "ProductRelated_Duration"
X_reg = df.drop(columns=[target_reg, "Revenue"])
y_reg = df[target_reg]
preprocessor_reg = get_preprocessor(X_reg)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

regressors = {
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "Regresión Lineal": LinearRegression()
}

reg_results = []
print("\n📈 Resultados de Regresión:\n")
for name, model in regressors.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor_reg),
        ("regressor", model)
    ])
    pipe.fit(X_train_reg, y_train_reg)
    y_pred = pipe.predict(X_test_reg)

    mae = mean_absolute_error(y_test_reg, y_pred)
    r2 = r2_score(y_test_reg, y_pred)
    reg_results.append((name, mae, r2))
    print(f"{name}: MAE = {mae:.2f}s, R² = {r2:.4f}")

best_reg = max(reg_results, key=lambda x: x[2])
print(f"\n✅ Mejor regresor: {best_reg[0]} (R² = {best_reg[2]:.4f})")


📈 Resultados de Regresión:

KNN Regressor: MAE = 577.04s, R² = 0.7231
Regresión Lineal: MAE = 476.83s, R² = 0.8007

✅ Mejor regresor: Regresión Lineal (R² = 0.8007)


### Integración en un MAS (Clasificación + Regresión)

**Sistema Multiagente que ejecuta simultáneamente:**
1. **Tareas de clasificación** (3 agentes):
   - SVM (kernel RBF, pesos balanceados)
   - Regresión Logística (pesos balanceados)
   - KNN Classifier (k=5)
   
   **Métricas:** Accuracy y F1-score  
   **Objetivo:** Predecir si el usuario realizará una compra (`Revenue`)

2. **Tareas de regresión** (2 agentes):
   - KNN Regressor (k=5)
   - Regresión Lineal
   
   **Métricas:** MAE y R²  
   **Objetivo:** Predecir tiempo en sección de productos (`ProductRelated_Duration`)

**Criterios de selección:**
- Mejor clasificador: Mayor F1-score
- Mejor regresor: Mayor R²

In [ ]:
print("🧠 DECISIÓN INTEGRADA DEL MAS:\n")

print("📌 Resultados de Clasificación:")
for name, acc, f1 in clf_results:
    print(f"   - {name}: Accuracy = {acc:.4f}, F1-score = {f1:.4f}")
print(f"\n✅ Mejor clasificador: {best_clf[0]} (F1-score = {best_clf[2]:.4f})\n")

print("📌 Resultados de Regresión:")
for name, mae, r2 in reg_results:
    print(f"   - {name}: MAE = {mae:.2f}s, R² = {r2:.4f}")
print(f"\n✅ Mejor regresor: {best_reg[0]} (R² = {best_reg[2]:.4f})")

🧠 DECISIÓN INTEGRADA DEL MAS:

📌 Resultados de Clasificación:
   - SVM: Accuracy = 0.8560, F1-score = 0.6077
   - Regresión Logística: Accuracy = 0.8500, F1-score = 0.6072
   - KNN Classifier: Accuracy = 0.8731, F1-score = 0.4943

✅ Mejor clasificador: SVM (F1-score = 0.6077)

📌 Resultados de Regresión:
   - KNN Regressor: MAE = 577.04s, R² = 0.7231
   - Regresión Lineal: MAE = 476.83s, R² = 0.8007

✅ Mejor regresor: Regresión Lineal (R² = 0.8007)


## Integración MAS: Análisis Conjunto de Clasificación y Regresión

### **Perspectiva de Clasificación (¿Compra el usuario?):**

El MAS ejecutó simultáneamente tres agentes de clasificación, donde SVM emergió como óptimo (F1-score: 0.6077). Este resultado supera a Regresión Logística (F1: 0.6072) y KNN (F1: 0.4943), por lo tanto proporciona identificación más confiable de usuarios con alta probabilidad de compra

### **Perspectiva de Regresión (¿Tiempo en productos?):**

En paralelo, dos agentes de regresión compitieron para predecir el **tiempo que un usuario pasará en la sección de productos** (`ProductRelated_Duration`), utilizando **Regresión Lineal** y **KNN**. El modelo **Regresión Lineal** fue seleccionado como el mejor, debido a su mayor precisión en las predicciones:

- **Regresión Lineal**: R² = 0.8007 | MAE = 476.83 segundos
- **KNN**: R² = 0.7231 | MAE = 577.04 segundos

Esto implica:

- **Regresión Lineal** superó a KNN en la métrica **R²**, lo que significa que el modelo es más capaz de explicar la variabilidad en los datos (el tiempo que los usuarios pasan en la sección de productos) en comparación con KNN.
- **MAE** (Error Absoluto Medio) de **476.83 segundos** frente a **577.04 segundos** en KNN indica que, en promedio, **Regresión Lineal** tiene una menor diferencia entre las predicciones y los valores reales de tiempo, lo que hace que sea más precisa en términos de tiempo.


### Decisión integrada del MAS:

- Selección óptima:

  - Clasificación: SVM para predecir compras

  - Regresión: Modelo lineal para estimar tiempo
